# Statistical income calculation

## Data

### Input

In [153]:
beneficiary = {
    "sexe": "homme",                   # Based on context
    "branche": "05-96",     # Occupation was in construction (CFC)
    "niveau_comp": 1,                  # CFC = skill level 2 for the original job
    "salaire_ofs": {
        "année": 2022,                 # The year we consult the ESS (or 2021,
        # depending on decision date)
        "salaire": 5305                # Will be determined from the ESS if needed
    },
    "ess": 2022,                       # Date of exigibilité: 01.07.2022 => relevant year for stats
    "salaire_effectif": {
        "salaire": 0,              # Actual salary pre-invalidity
        "année": 0
    },
    "horaire": 100,                      # Now working only 50%
    "diminution": 50,                # Hourly wage reduction (if known)
    "abattement": None,                # E.g. if capacity ≤50% might get a 10% deduction;
    # depends on medical/SMR input
    "salaire_as": {
        "salaire": 90000,              # Salary before the health issue
        "année": 2021
    }
}


### Database

In [112]:
import pandas as pd

# Create the DataFrame from the cleaned data
data = {
    'id': ['05-96', '01-03', '05-43', '05-09', '35-39', '10-33', '10-12', '16-18', '19-21', '22-23', '24-25', '26-27', '28-30', '31-33', '41-43', '45-96', '45-47', '45', '46', '47', '49-53', '49-52', '53', '55-56', '58-63', '58-61', '62-63', '64-66', '64/66', '65', '69-75', '77-82', '84', '86-88', '90-96'],
    'label': ['TOTAL', 'SECTEUR PRIMAIRE', 'SECTEUR SECONDAIRE', 'Industries extractives, production et distribution d’énergie et d’eau', 'gestion des déchets', 'Industries manufacturières', 'Industries alimentaires et du tabac', 'Industries du bois et du papier ; imprimerie', 'Cokéfaction et raffinage, Industrie chimique et pharmaceutique', 'Industries du caoutchouc, du plastique et produits minéraux non métalliques', 'Fabrication de produits métalliques', 'Fabrication de produits informatiques et électroniques, d’équipements électriques, optique, horlogerie', 'Fabrication de machines, équipements et matériels de transport', 'Autres industries manufacturières; réparation et installation', 'Construction', 'SECTEUR TERTIAIRE', 'Commerce', 'Commerce et réparation d’automobiles et de motocycles', 'Commerce de gros', 'Commerce de détail', 'Transports et courrier', 'Transports et entreposage', 'Activités de poste et de courrier', 'Hébergement et restauration', 'Edition, diffusion, télécommunications, activités informatiques', 'Édition, audiovisuel et diffusion, télécommunications', 'Activités informatiques et services d’information', 'Activités financières et assurance', 'Activités des services financiers, activités auxiliaires de services financiers et d’assurance', 'Assurance', 'Activités spécialisées, scientifiques et techniques', 'Activités de services administratifs et de soutien', 'Administration publique', 'Santé, hébergement médico-social et action sociale', 'Arts, spectacles et activités récréatives, autres activités de services'],
    '2011':[100.9513, 100.9513, 100.9544, 101.4944, 101.4944, 100.9002, 100.2127, 100.8089, 101.4288, 101.1103, 100.8698, 100.5908, 101.0703, 101.4913, 101.0127, 100.95, 101.3666, 101.0275, 101.0006, 101.7282, 100.6464, 100.67, 100.5518, 99.9894, 101.7023, 100.7542, 102.452, 101.4209, 101.1314, 102.2603, 100.8398, 100.6379, 100.0245, 100.8762, 100.3936],
    '2012':[101.7977, 101.7977, 101.6171, 101.4975, 101.4975, 101.5734, 100.6443, 101.5147, 102.8733, 102.0455, 101.1133, 100.9305, 102.0073, 102.0918, 101.7447, 101.8678, 102.2374, 101.0469, 101.8337, 102.8017, 101.4937, 101.3521, 102.0623, 102.3798, 102.1077, 101.4441, 102.6324, 102.3487, 101.8594, 103.7673, 102.1111, 101.4828, 100.9158, 101.1692, 102.0683],
    '2013':[102.5511, 102.5511, 102.2865, 101.5358, 101.5358, 102.3384, 100.6763, 102.0552, 103.9178, 102.473, 101.4371, 101.9166, 103.6206, 102.201, 102.2641, 102.6539, 102.8902, 101.9497, 101.8127, 103.9428, 102.0654, 101.9342, 102.5917, 102.737, 103.1973, 101.6927, 104.387, 103.1463, 102.9023, 103.8536, 104.1649, 102.4918, 102.0048, 101.682, 102.0439],
    '2014':[103.348, 103.348, 103.2236, 102.0162, 102.0162, 103.4723, 102.0072, 102.2616, 104.7204, 104.3637, 102.9801, 103.5659, 104.2468, 102.9054, 102.7717, 103.3963, 103.6695, 103.2377, 102.8204, 104.4363, 101.6702, 101.1933, 103.5841, 103.8655, 104.2868, 102.5758, 105.6396, 104.5246, 104.2736, 105.2521, 104.4405, 103.5097, 102.3699, 101.7356, 104.5736],
    '2015':[103.7277, 103.7277, 103.6934, 102.9235, 102.9235, 104.2031, 102.9146, 102.5426, 105.886, 103.6205, 104.1806, 104.4205, 104.3366, 104.1815, 102.5207, 103.7409, 104.13, 103.8963, 102.9011, 105.1603, 102.3412, 101.7418, 104.7466, 104.1656, 104.3264, 102.743, 105.5783, 105.085, 104.9778, 105.3959, 104.185, 103.3165, 102.5003, 102.0771, 105.0915],
    '2016':[104.429, 104.429, 104.1506, 102.9235, 102.9235, 104.6776, 102.7528, 101.824, 107.4631, 103.8313, 104.095, 105.1019, 105.2622, 104.5834, 102.9352, 104.5378, 105.0675, 103.4392, 104.6611, 105.6409, 102.4057, 101.834, 104.7451, 105.1351, 104.851, 103.2522, 106.1147, 106.517, 106.6755, 105.854, 105.396, 103.5151, 103.231, 102.6098, 106.3462],
    '2017':[104.8461, 104.8461, 104.5835, 103.6518, 103.6518, 105.1888, 102.9497, 103.158, 107.6334, 103.559, 103.9229, 105.9405, 105.9566, 105.6708, 103.2319, 104.9488, 105.4629, 104.1101, 105.0772, 105.9379, 102.7543, 102.1535, 105.1672, 105.4501, 105.8593, 104.4621, 106.9739, 106.9689, 107.057, 106.563, 106.0379, 103.9738, 103.3703, 102.9363, 106.3561],
    '2018':[105.3519, 105.3519, 104.9281, 103.4385, 103.4385, 105.489, 102.9044, 102.7292, 109.4953, 104.7117, 104.4283, 105.6714, 105.8776, 105.8641, 103.7855, 105.5177, 106.3892, 105.3808, 105.7325, 107.043, 102.6406, 102.0564, 105.0124, 105.8577, 107.3237, 105.9515, 108.4209, 108.4819, 108.6563, 107.7592, 105.8882, 104.1295, 103.6917, 103.5323, 106.2203],
    '2019':[106.3124, 106.3124, 105.8385, 104.9755, 104.9755, 106.3043, 103.6752, 104.4201, 109.5667, 105.2472, 104.3814, 107.4541, 106.8798, 105.3344, 104.7761, 106.4978, 106.7478, 104.7376, 106.6944, 107.0615, 103.8623, 103.1797, 106.4825, 105.057, 107.9037, 106.1939, 109.2522, 110.2176, 110.3948, 109.4833, 107.6549, 104.483, 104.5212, 103.9607, 0.0],
    '2020':[107.2054242, 107.2054242, 106.4206118, 103.1174337, 103.1174337, 106.9740171, 104.193576, 104.0859557, 109.5119167, 105.7629113, 106.6464764, 108.2277695, 107.5745187, 105.6820035, 105.6457416, 107.4988793, 107.3242381, 106.9580371, 106.0328947, 108.3569442, 103.5091682, 102.7669812, 106.2801833, 0.0, 110.5149695, 108.3283974, 112.2457103, 109.2917722, 109.8317865, 107.1841507, 110.5185203, 105.3920021, 104.5943648, 105.738428, 106.4964728],
    '2021':[107, 107, 105.9, 103.6, 103.6, 106.1, 104.5, 103.9, 106.1, 106.3, 105.7, 107, 106.5, 107.9, 105.7, 107.4, 107.3, 106.5, 106.3, 108, 103.7, 103.1, 105.7, 105.3, 108.6, 109.1, 108.8, 107.7, 108, 106.5, 109.7, 105.9, 106.8, 105.6, 102.9],
    '2022':[108, 108, 106.7, 104.8, 104.8, 106.9, 104.5, 104.8, 110.4, 104, 105.8, 106.4, 108.6, 108.6, 106.2, 108.5, 108.5, 107.3, 107.8, 108.9, 104.4, 104, 105.4, 106.1, 110.5, 109.7, 111.5, 109.7, 109.7, 109.4, 111.4, 107.4, 107.5, 106.5, 101.5],
    '2023':[109.8, 109.8, 109, 107.2, 107.2, 109.1, 106.5, 106.8, 111.3, 106.5, 108.9, 109.4, 111.4, 108.2, 108.7, 110.2, 110.3, 110, 109.3, 110.7, 106.5, 106.1, 107.4, 107.9, 112.5, 112.2, 113.3, 110.9, 110.8, 111.1, 110.7, 110.4, 111.4, 106.6, 104.5],
    '2024':[111.63, 111.63, 111.3495783, 109.6549618, 109.6549618, 111.345276, 108.5382775, 108.8381679, 112.207337, 109.0600962, 112.0908318, 112.4845865, 114.2721915, 107.8014733, 111.2588512, 111.9266359, 112.1298618, 112.7679404, 110.820872, 112.5297521, 108.6422414, 108.2424038, 109.4379507, 109.7305372, 114.5361991, 114.7569736, 115.1290583, 112.1131267, 111.9110301, 112.8264168, 110.0043986, 113.4837989, 115.4414884, 106.7000939, 107.58867],
    '2025':[113.4905, 113.4905, 113.7498035, 112.1661442, 112.1661442, 113.6367597, 110.6155651, 110.9152322, 113.1220707, 111.6817331, 115.3751567, 115.6561444, 117.2184359, 107.4044145, 113.8779391, 113.6803252, 113.9900807, 115.6055307, 112.3629064, 114.389748, 110.8275738, 110.4280678, 111.5145721, 111.5921298, 116.6092525, 117.3722191, 116.987644, 113.3395237, 113.0332008, 114.579661, 109.3131681, 116.6537374, 119.6295982, 106.8002818, 110.7686306],
    '2026':[115.3820083, 115.3820083, 116.2017674, 114.7348345, 114.7348345, 115.975402, 112.7326094, 113.0319351, 114.0442615, 114.3663901, 118.7557142, 118.9171259, 120.2406423, 107.0088181, 116.5586815, 115.4614916, 115.8811603, 118.5145235, 113.9263977, 116.2804876, 113.0568641, 112.6578653, 113.6305982, 113.4853044, 118.7198272, 120.0470646, 118.8762338, 114.5793362, 114.166624, 116.3601493, 108.626281, 119.9122217, 123.9696487, 106.9005637, 114.0425803],
    '2027':[117.3050418, 117.3050418, 118.7065852, 117.3623498, 117.3623498, 118.3621736, 114.8901713, 115.1890331, 114.9739701, 117.1155822, 122.235324, 122.2700524, 123.3407694, 106.6146788, 119.3025299, 117.2705656, 117.8036128, 121.4967156, 115.5116444, 118.2024791, 115.3309964, 114.9326876, 115.7867765, 115.4105971, 120.8686023, 122.7828682, 120.795312, 115.8327109, 115.3114124, 118.1683052, 107.9437101, 123.2617251, 128.4671523, 107.0009398, 117.413297],
    '2028':[119.2601258, 119.2601258, 121.2653963, 120.0500372, 120.0500372, 120.798065, 117.0890263, 117.3872971, 115.9112579, 119.9308606, 125.8168883, 125.7175162, 126.5208261, 106.2219912, 122.1109699, 119.1079846, 119.7579584, 124.5539489, 117.1189493, 120.1562391, 117.6508728, 117.2534438, 117.983869, 117.3685525, 123.0562693, 125.5810193, 122.7453708, 117.0997961, 116.46768, 120.0045585, 107.2654283, 126.70479, 133.127821, 107.1014102, 120.8836407],
    '2029':[121.2477946, 121.2477946, 123.8793646, 122.7992747, 122.7992747, 123.2840869, 119.3299646, 119.6275127, 116.8561867, 122.813814, 129.5033945, 129.262183, 129.7828731, 105.83075, 124.9855219, 120.9741927, 121.7447264, 127.6881116, 118.7486192, 122.1422927, 120.0174133, 119.6210614, 120.2226521, 119.359725, 125.2835321, 128.4429386, 124.7269104, 118.380742, 117.6355419, 121.869346, 106.5914085, 130.24403, 137.9575745, 107.2019749, 124.4565562],
    '2030':[123.2685912, 123.2685912, 126.5496789, 125.6114718, 125.6114718, 125.8212711, 121.6137917, 121.9104805, 117.8088186, 125.7660691, 133.2979174, 132.9067935, 133.1290246, 105.4409498, 127.9277423, 122.8696409, 123.7644546, 130.9011396, 120.4009655, 124.1611735, 122.4315567, 122.0364867, 122.5039169, 121.3846779, 127.5511074, 131.3700794, 126.740439, 119.6756999, 118.8151143, 123.763111, 105.9216241, 133.8821314, 142.962547, 107.3026341, 128.1350751],
    '2031':[125.3230677, 125.3230677, 129.2775539, 128.4880704, 128.4880704, 128.4106705, 123.9413283, 124.2370164, 118.7692166, 128.789292, 137.203622, 136.6541655, 136.5614488, 105.0525854, 130.939224, 124.7947873, 125.8176898, 134.1950173, 122.0763036, 126.2134243, 124.8942604, 124.5006849, 124.8284694, 123.4439844, 129.8597247, 134.363928, 128.786473, 120.9848234, 120.0065147, 125.6863038, 105.2560484, 137.6218557, 148.1490952, 107.4033877, 131.9223187],
    '2032':[127.4117855, 127.4117855, 132.0642303, 131.4305453, 131.4305453, 131.0533597, 126.3134112, 126.6079518, 119.7374439, 131.8851884, 141.2237659, 140.5071964, 140.0823701, 104.6656513, 134.0215975, 126.7500973, 127.9049878, 137.5717792, 123.7749535, 128.2995966, 127.4065013, 127.0146411, 127.197131, 125.5382273, 132.210127, 137.4260048, 130.8655372, 122.3082672, 121.2098617, 127.6393816, 104.5946549, 141.4660416, 153.5238066, 107.5042359, 135.8215006],
    '2033':[129.5353152, 129.5353152, 134.9109757, 134.4404051, 134.4404051, 133.7504354, 128.7308927, 129.0241341, 120.7135644, 135.0555054, 145.3617023, 144.4688655, 143.6940702, 104.2801425, 137.1765315, 128.7360436, 130.0269139, 141.0335108, 125.4972395, 130.4202511, 129.9692758, 129.5793598, 129.6107388, 127.6679993, 134.6030704, 140.5578645, 132.9781647, 123.6461881, 122.4252751, 129.6228089, 103.9374174, 145.417607, 159.0935075, 107.6051789, 139.8359291],
    '2034':[131.6942372, 131.6942372, 137.8190848, 137.519193, 137.519193, 136.5030168, 131.1946419, 131.4864267, 121.6976423, 138.302032, 149.6208826, 148.5422358, 147.3988897, 103.8960536, 140.4057342, 130.753106, 132.1840424, 144.5823503, 127.2434905, 132.5759578, 132.5836003, 132.1958661, 132.0701456, 129.8339031, 137.0393251, 143.7610975, 135.1248974, 124.9987444, 123.6528758, 131.6370573, 103.2843097, 149.4795514, 164.8652719, 107.7062166, 143.9690108]
}

t1_10 = pd.DataFrame(data)

In [113]:
t1_rai_ids = data['id']
t1_labels = data['label']

In [114]:
t1 = ['B-S', 'B-S', 'B-F', 'B, D, E', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'B, D, E', 'B, D, E', 'F', 'F', 'F', 'G-S', 'G', 'G', 'G', 'G', 'H', 'H', 'H', 'H', 'H', 'I', 'I', 'I', 'J', 'J', 'J', 'J', 'K', 'K', 'K', 'K', 'G-S', 'M', 'M', 'M', 'M', 'M', 'M', 'N', 'N', 'N', 'O', 'G-S', 'Q', 'Q', 'Q', 'Q', 'R, S', 'R, S']

In [115]:
data = {
    'id':['05-96', '01-03', '05-43', '05-09/35-39', '10-33', '41-43', '45-96', '45-47', '49-53', '55/56', '58-63', '64-66', '69-75', '77-82', '84', '86-88', '90-96'],
    'label':['TOTAL', 'SECTEUR PRIMAIRE', 'SECTEUR SECONDAIRE', 'Industries extractives, production et distribution d\'électricité, de gaz et d\'eau', 'Industries manufacturières', 'Construction', 'SECTEUR TERTIAIRE', 'Commerce, réparation d\'automobiles et motocycles', 'Transports et entreposage, Poste et courrier', 'Hébergement et restauration', 'Infomation et communication', 'Activités financières et d\'assurance', 'Activités spécialisées scientifiques et techniques', 'Activités de services administratifs et de soutien', 'Administration publique', 'Santé, Hébergement médico-social et action sociale', 'Arts, spectacles et activités récréatives, autres activités de services'],
    '2011':[100.9507, 100.9507, 100.932, 101.4944, 100.8523, 101.0127, 100.9615, 101.1856, 100.4811, 99.9922, 101.8357, 101.3906, 100.7836, 100.4953, 100.0238, 101.2454, 100.7361],
    '2012':[101.7204, 101.7204, 101.5569, 101.4975, 101.4686, 101.7447, 101.8145, 101.9118, 101.3914, 101.8625, 102.1845, 102.1049, 102.0252, 101.2934, 101.0435, 101.5172, 102.2561],
    '2013':[102.5036, 102.5036, 102.2245, 101.5358, 102.2539, 102.2641, 102.6642, 102.4448, 101.8862, 102.6123, 103.4038, 102.9317, 104.1018, 102.4196, 101.9874, 102.2603, 102.2013],
    '2014':[103.2055, 103.2055, 103.093, 102.0162, 103.3277, 102.7717, 103.2703, 102.8832, 101.4266, 103.4042, 104.4703, 104.2376, 104.0317, 103.3892, 102.3431, 102.5398, 104.2172],
    '2015':[103.517, 103.517, 103.501, 102.9235, 104.0246, 102.5207, 103.5261, 103.1559, 102.2015, 103.6738, 104.4914, 104.568, 103.7791, 103.1831, 102.3443, 102.8793, 104.7418],
    '2016':[104.1277, 104.1277, 103.9244, 103.3991, 104.4466, 102.9352, 104.2452, 103.908, 102.2753, 104.6758, 104.9184, 105.9897, 105.0352, 103.31, 103.0715, 102.9724, 106.0058],
    '2017':[104.5568, 104.5568, 104.339, 103.6518, 104.9474, 103.2319, 104.6827, 104.3653, 102.6246, 104.9632, 105.9014, 106.2897, 105.6714, 103.6962, 103.233, 103.4962, 105.8772],
    '2018':[105.0719, 105.0719, 104.699, 103.4385, 105.2621, 103.7855, 105.2872, 104.9985, 102.6009, 105.2834, 107.3608, 107.8018, 105.2874, 103.9527, 104.2472, 104.7731, 104.8941],
    '2019':[105.9765, 105.9765, 105.484, 104.9755, 105.8408, 104.7761, 106.261, 105.0439, 104.0385, 104.5189, 107.9316, 109.5266, 107.4547, 104.2823, 105.0812, 104.2752, 0.0],
    '2020':[106.8400503, 106.8400503, 106.1425788, 103.1168345, 106.6757093, 105.6439403, 107.23515, 105.2140644, 103.3340899, 0.0, 110.0136637, 108.9020351, 110.7359822, 105.0810847, 105.4162998, 107.7711626, 103.4103094],
    '2021':[106, 106, 105.5, 103.6, 105.6, 105.7, 106.4, 105.1, 103.4, 104.6, 108.1, 106, 109.2, 105.5, 105.6, 106.3, 100.7],
    '2022':[107.1, 107.1, 106.1, 104.8, 106.3, 106.2, 107.7, 106.3, 103.9, 105.3, 110, 108.3, 110.7, 107.2, 106.6, 107.9, 100.5],
    '2023':[108.9, 108.9, 108.4, 107.2, 108.5, 108.7, 109.2, 107.7, 106, 107.4, 111.9, 110.8, 110.2, 109.1, 109, 107.2, 103.5],
    '2024':[110.7302521, 110.7302521, 110.7498586, 109.6549618, 110.7455315, 111.2588512, 110.7208914, 109.1184384, 108.1424447, 109.5418803, 113.8328182, 113.3577101, 109.7022584, 111.0336754, 111.4540338, 106.5045412, 106.5895522],
    '2025':[112.5912647, 112.5912647, 113.1506567, 112.1661442, 113.0375369, 113.8779391, 112.2629651, 110.555558, 110.3281919, 111.7264762, 115.7990214, 115.9744624, 109.2067649, 113.001623, 113.9633178, 105.8135943, 109.7713299],
    '2026':[114.4835549, 114.4835549, 115.6034984, 114.7348345, 115.3769779, 116.5586815, 113.8265161, 112.0116048, 112.5581168, 113.9546396, 117.7991863, 118.6516199, 108.7135094, 115.0044503, 116.5290961, 105.1271298, 113.048086],
    '2027':[116.4076483, 116.4076483, 118.1095121, 117.3623498, 117.7648363, 119.3025299, 115.4118436, 113.4868282, 114.8331124, 116.2272392, 119.8338995, 121.3905769, 108.2224818, 117.0427754, 119.1526405, 104.4451188, 116.4226558],
    '2028':[118.3640793, 118.3640793, 120.6698502, 120.0500372, 120.2021142, 122.1109699, 117.0192509, 114.9814807, 117.1540897, 118.5451614, 121.9037578, 124.1927601, 107.733672, 119.1172276, 121.8352515, 103.7675323, 119.8979589],
    '2029':[120.3533916, 120.3533916, 123.2856905, 122.7992747, 122.6898344, 124.9855219, 118.6490455, 116.4958182, 119.5219779, 120.9093099, 124.0093682, 127.059629, 107.2470701, 121.2284471, 124.5782591, 103.0943416, 123.4770025],
    '2030':[122.3761377, 122.3761377, 125.9582361, 125.6114718, 125.2290407, 127.9277423, 120.3015392, 118.0300999, 121.9377253, 123.3206067, 126.1513482, 129.9926768, 106.762666, 123.3770856, 127.3830229, 102.4255183, 127.1628831],
    '2031':[124.4328795, 124.4328795, 128.6887163, 128.4880704, 127.8207989, 130.939224, 121.9770481, 119.5845885, 124.4022992, 125.779992, 128.330326, 132.9934311, 106.2804498, 125.5638063, 130.2509333, 101.7610339, 130.9587901],
    '2032':[126.5241884, 126.5241884, 131.4783868, 131.4305453, 130.4661964, 134.0215975, 123.6758928, 121.1595502, 126.9166863, 128.2884249, 130.5469407, 136.0634549, 105.8004116, 127.7892843, 133.1834121, 101.1008604, 134.8680077],
    '2033':[128.6506453, 128.6506453, 134.3285309, 134.4404051, 133.1663434, 137.1765315, 125.3983982, 122.7552545, 129.4818937, 130.8468835, 132.8018424, 139.2043472, 105.3225416, 130.0542063, 136.181913, 100.4449697, 138.8939184],
    '2034':[130.812841, 130.812841, 137.2404595, 137.519193, 135.9223731, 140.4057342, 127.144894, 124.3719747, 132.0989483, 133.4563655, 135.0956924, 142.417744, 104.8468301, 132.3592715, 139.2479223, 99.79333412, 143.0400055]
}
t1_1_10 = pd.DataFrame(data)

In [116]:
t1_hf_ids = data['id']
t1_hf_labels = data['label']

In [117]:
data = {
    'id': t1_hf_ids,
    'label': t1_hf_labels,
    '2011':[100.9523, 100.9523, 101.1001, 100, 101.1001, 100, 100.9339, 101.6251, 101.2597, 99.9867, 100.8421, 101.4783, 100.9488, 100.8996, 100.0257, 100.7337, 100.0547],
    '2012':[101.9506, 101.9506, 102.0095, 100, 102.0095, 100, 101.9432, 102.702, 101.8735, 102.8839, 101.6124, 102.811, 102.2778, 101.8304, 100.6535, 101.0349, 101.8824],
    '2013':[102.645, 102.645, 102.6903, 100, 102.6903, 100, 102.6393, 103.5257, 102.7299, 102.8586, 101.8662, 103.5531, 104.2874, 102.6242, 102.0404, 101.4587, 101.8881],
    '2014':[103.6299, 103.6299, 104.0743, 100, 104.0743, 100, 103.5744, 104.7917, 102.5739, 104.3152, 103.1034, 105.0687, 105.2339, 103.7307, 102.4248, 101.4252, 104.9264],
    '2015':[104.1446, 104.1446, 104.9462, 100, 104.9462, 100, 104.0445, 105.52, 102.8591, 104.645, 103.262, 106.0653, 104.9727, 103.5613, 102.8207, 101.7675, 105.4376],
    '2016':[105.0263, 105.0263, 105.623, 100, 105.623, 100, 104.9521, 106.7342, 102.8941, 105.5722, 104.4783, 107.5164, 106.0953, 103.9477, 103.5577, 102.4576, 106.6824],
    '2017':[105.4197, 105.4197, 106.1748, 100, 106.1748, 100, 105.3255, 107.0368, 103.2405, 105.9215, 105.6662, 108.2415, 106.7485, 104.5965, 103.6615, 102.7133, 106.8362],
    '2018':[105.9071, 105.9071, 106.4192, 100, 106.4192, 100, 105.8436, 108.4006, 102.8362, 106.4387, 107.1685, 109.7571, 107.0577, 104.4871, 102.8035, 103.0652, 107.58],
    '2019':[106.9787, 106.9787, 108.1452, 100, 108.1452, 100, 106.8326, 109.2272, 103.3614, 105.5934, 107.8146, 111.5133, 108.0392, 104.9033, 103.626, 103.8255, 0.0],
    '2020':[107.9195981, 107.9195981, 108.295042, 100, 108.295042, 100, 107.8576222, 110.2826411, 103.9249832, 0.0, 113.4820209, 110.084843, 110.2645463, 106.0042576, 103.3713237, 105.1179836, 109.2792845],
    '2021':[108.6, 108.6, 108.2, 100, 108.2, 100, 108.6, 110.3, 104.5, 105.9, 112, 110.6, 110.5, 106.6, 107.2, 105.4, 105.1],
    '2022':[109.4, 109.4, 109.5, 100, 109.5, 100, 109.4, 111.5, 105.7, 106.7, 113.8, 112, 112.4, 107.7, 107.7, 106.1, 102.8],
    '2023':[111.3, 111.3, 111.6, 100, 111.6, 100, 111.3, 113.8, 108, 108.2, 116.5, 111.1, 111.5, 112.4, 112.7, 106.4, 105.7],
    '2024':[113.2329982, 113.2329982, 113.740274, 100, 113.740274, 100, 113.2329982, 116.1474439, 110.3500473, 109.7210872, 119.2640598, 110.2072321, 110.6072064, 117.3051068, 117.9321263, 106.7008483, 108.6818093],
    '2025':[115.1995676, 115.1995676, 115.9215943, 100, 115.9215943, 100, 115.1995676, 118.5433105, 112.7512309, 111.2635579, 122.0936991, 109.3216383, 109.7215615, 122.4242711, 123.4071554, 107.0025472, 111.7477359],
    '2026':[117.2002914, 117.2002914, 118.1447482, 100, 118.1447482, 100, 117.2002914, 120.9885985, 115.2046636, 112.8277129, 124.9904741, 108.4431609, 108.8430081, 127.7668345, 129.136364, 107.3050991, 114.9001525],
    '2027':[119.2357626, 119.2357626, 120.4105378, 100, 120.4105378, 100, 119.2357626, 123.4843275, 117.7114822, 114.413857, 127.9559774, 107.5717426, 107.9714893, 133.3425459, 135.1315527, 107.6085066, 118.1414993],
    '2028':[121.3065848, 121.3065848, 122.719781, 100, 122.719781, 100, 121.3065848, 126.0315378, 120.2728484, 116.0222992, 130.9918398, 106.7073268, 107.1069489, 139.16158, 141.4050696, 107.9127719, 121.4742847],
    '2029':[123.4133719, 123.4133719, 125.0733111, 100, 125.0733111, 100, 123.4133719, 128.6312915, 122.8899492, 117.6533531, 134.0997305, 105.8498572, 106.249331, 145.2345552, 147.969836, 108.2178976, 124.9010885],
    '2030':[125.5567486, 125.5567486, 127.4719773, 100, 127.4719773, 100, 125.5567486, 131.2846724, 125.5639973, 119.3073365, 137.2813586, 104.999278, 105.3985802, 151.5725534, 154.8393734, 108.523886, 128.4245628],
    '2031':[127.7373503, 127.7373503, 129.9166454, 100, 129.9166454, 100, 127.7373503, 133.9927867, 128.2962318, 120.9845717, 140.5384734, 104.1555338, 104.5546413, 158.1871402, 162.0278309, 108.8307396, 132.0474347],
    '2032':[129.9558234, 129.9558234, 132.4081975, 100, 132.4081975, 100, 129.9558234, 136.7567635, 131.087919, 122.6853858, 143.872866, 103.3185697, 103.7174601, 165.0903859, 169.5500143, 109.1384608, 135.7725082],
    '2033':[132.2128258, 132.2128258, 134.9475328, 100, 134.9475328, 100, 132.2128258, 139.577755, 133.9403524, 124.41011, 147.2863699, 102.4883312, 102.8869822, 172.2948874, 177.421417, 109.4470521, 139.6026665],
    '2034':[134.5090267, 134.5090267, 137.5355677, 100, 137.5355677, 100, 134.5090267, 142.4569374, 136.8548539, 126.1590807, 150.780862, 101.6647643, 102.063154, 179.8137915, 185.6582516, 109.756516, 143.5408741]
}
t1_2_10 = pd.DataFrame(data)

In [118]:
data = {
    'id': ['01-96', '01-03', '05-43', '5-9', '10-33', '10-12', '13-15', '16-18', '19-20', '21', '22-23', '24-25', '26', '27', '28', '29-30', '31-33', '35', '36-39', '41-43', '41-42', '43', '45-96', '45-47', '45', '46', '47', '49-53', '49', '50-51', '52', '53', '55-56', '55', '56', '58-63', '58-60', '61', '62-63', '64-66', '64', '65', '66', '68', '69-75', '69', '70', '71', '72', '73-75', '77-82', '77+79-82', '78', '84', '85', '86-88', '86', '87', '88', '90-93', '94-96'],
    'label': [ 'TOTAL', 'SECTEUR PRIMAIRE', 'SECTEUR SECONDAIRE', 'Industries extractives', 'Industrie manufacturière', 'Industries alimentaires et du tabac', 'Industries du textile et de l’habillement', 'Industries du bois et du papier ; imprimerie', 'Cokéfaction, raffinage et industrie chimique', 'Industrie pharmaceutique', 'Industries du caoutchouc et du plastique', 'Fabrication de produits métalliques', 'Fabrication de produits électroniques; horlogerie', 'Fabrication d’équipements électriques', 'Fabrication de machines et équipements n.c.a', 'Fabrication de matériels de transport', 'Autres industries manufacturières; rép. et inst.', 'Production et distribution d’énergie', 'Production et distr. d’eau; gestion des déchets', 'Construction', 'Construction de bâtiments et génie civil', 'Travaux de construction spécialisés', 'SECTEUR TERTIAIRE', 'Commerce; réparation d\'automobiles et de motocycles', 'Commerce et rép. d\’automobiles et de motocycles', 'Commerce de gros', 'Commerce de détail', 'Transport et entreposage', 'Transports terrestres et transport par conduites', 'Transports par eau, transports aériens', 'Entreposage et services auxiliaires des transports', 'Activités de poste et de courrier', 'Hébergement et restauration', 'Hébergement', 'Restauration', 'Information et communication', 'Édition, audiovisuel et diffusion', 'Télécommunications', 'Activités informatiques et services d’information', 'Activités financières et d\'assurance', 'Activités des services financiers', 'Assurance', 'Activités aux. de services financiers et d’assurance', 'Activités immobilières', 'Activités spécialisées, scientifiques et techniques', 'Activités juridiques et comptables', 'Activités des sièges sociaux ; conseil de gestion', 'Activités d’architecture et d’ingénierie', 'Recherche-développement scientifique', 'Autres activités spécialisées, scient. et techn.', 'Activités de services administratifs et de soutien', 'Activités de services administratifs (sans 78)', 'Activités liées à l\'emploi', 'Administration publique', 'Enseignement', 'Santé humaine et action sociale', 'Activités pour la santé humaine', 'Hébergement médico-social et social', 'Action sociale sans hébergement', 'Arts, spectacles et activités récréatives', 'Autres activités de services'],
    '2004':[41.67362417, 42.98353643, 41.38606498, 42.25962291, 41.1978938, 41.92822095, 41.74138854, 41.32149495, 40.84737702, 40.7117, 41.77699998, 41.38346255, 40.56443, 40.81595, 40.92367, 41.31536993, 41.32787366, 41.13106, 42.92422469, 41.7570494, 42.13490223, 41.54154, 41.67767035, 41.79757357, 42.36605, 41.83609, 41.64347, 42.07119101, 42.0609, 41.28301055, 42.45593, 41.98835, 42.12635078, 42.08624, 42.14803, 40.86358488, 40.60954953, 40.30827, 41.22553082, 41.43349982, 41.51494, 41.32068, 41.30154, 41.52619, 41.40727731, 41.19059, 41.23719, 41.72655, 40.65625, 41.51462571, 42.15010296, 42.16511317, 41.93972, 41.58442, 41.3973, 41.5898828, 41.6974, 41.50078, 41.27713, 41.5290549, 41.92382326],
    '2005':[41.66915863, 42.90770405, 41.41005214, 42.39040938, 41.21886768, 41.98592218, 41.63643933, 41.37213127, 40.93403616, 40.6914, 41.75778081, 41.38970145, 40.54156, 40.96892, 40.95473, 41.28980995, 41.41656607, 41.18051, 42.84489557, 41.75962854, 42.10858043, 41.57273, 41.67634407, 41.81196842, 42.32533, 41.93867, 41.60469, 42.2186238, 42.24345, 41.87317159, 42.52475, 41.98022, 42.10678258, 42.08094, 42.11944, 40.85793498, 40.58227195, 40.32146, 41.20298765, 41.51870003, 41.61767, 41.35377, 41.4412, 41.46928, 41.36536376, 41.22133, 41.20091, 41.72961, 40.54019, 41.36839122, 42.12838633, 42.14118515, 41.93916, 41.41561, 41.4292, 41.56011327, 41.67379, 41.47712, 41.24154, 41.50132716, 41.92118416],
    '2006':[41.66915863, 42.90770405, 41.41005214, 42.39040938, 41.21886768, 41.98592218, 41.63643933, 41.37213127, 40.93403616, 40.6914, 41.75778081, 41.38970145, 40.54156, 40.96892, 40.95473, 41.28980995, 41.41656607, 41.18051, 42.84489557, 41.75962854, 42.10858043, 41.57273, 41.67634407, 41.81196842, 42.32533, 41.93867, 41.60469, 42.2186238, 42.24345, 41.87317159, 42.52475, 41.98022, 42.10678258, 42.08094, 42.11944, 40.85793498, 40.58227195, 40.32146, 41.20298765, 41.51870003, 41.61767, 41.35377, 41.4412, 41.46928, 41.36536376, 41.22133, 41.20091, 41.72961, 40.54019, 41.36839122, 42.12838633, 42.14118515, 41.93916, 41.41561, 41.4292, 41.56011327, 41.67379, 41.47712, 41.24154, 41.50132716, 41.92118416],
    '2007':[41.66239409, 42.79091253, 41.39564303, 42.66027766, 41.21590425, 42.11774373, 41.73890791, 41.40619296, 40.82141717, 40.66229, 41.7266758, 41.36359986, 40.51381, 41.06375, 40.94236, 41.13240086, 41.33628057, 41.1717, 42.94408492, 41.71057162, 42.01499973, 41.54752, 41.67995571, 41.82989538, 42.33364, 41.92995, 41.64109, 42.44163976, 42.70466, 41.89184579, 42.60975, 41.97724, 42.0767957, 42.08869, 42.07097, 40.94984114, 40.73098279, 40.2392, 41.33534461, 41.44080139, 41.61839, 41.17064, 41.25364, 41.52851, 41.36904597, 41.12623, 41.30438, 41.66829, 40.49591, 41.51278904, 42.03864229, 42.05385144, 41.81378, 41.33093, 41.43892, 41.51982852, 41.54389, 41.64947, 41.06314, 41.55053788, 41.94097919],
    '2008':[41.62834008, 42.70115365, 41.34829206, 42.64008055, 41.19084549, 42.1072246, 41.64571292, 41.41675646, 40.86282715, 40.64049, 41.7544923, 41.38529947, 40.48665, 40.94695, 41.00812, 41.09935529, 41.28323675, 41.251, 42.73955403, 41.62215599, 41.93556659, 41.46111, 41.66468944, 41.86233054, 42.33344, 41.91276, 41.71296, 42.38335564, 42.73144, 41.36753133, 42.35404, 42.02047, 42.02750157, 42.05585, 42.01354, 40.96382876, 40.70758594, 40.20703, 41.31391874, 41.37873736, 41.47628, 41.27784, 41.18717, 41.56683, 41.43423242, 41.14272, 41.41156, 41.75729, 40.59079, 41.54244786, 42.00990234, 42.03223597, 41.72076, 41.32079, 41.21681, 41.56597308, 41.60641, 41.68328, 41.12525, 41.53497107, 41.93734164],
    '2009':[41.63949694, 42.98309374, 41.34252738, 42.57016896, 41.1654233, 42.14922049, 41.67782262, 41.34894618, 40.82485432, 40.60529, 41.72182774, 41.29478127, 40.47838, 40.90425, 40.90946, 41.29688741, 41.42038459, 41.14385, 43.01770743, 41.65548967, 41.98457348, 41.48639, 41.66574002, 41.91708508, 42.31086, 41.89925, 41.82932, 42.36724923, 42.8337, 41.61464761, 42.14536, 41.91569, 42.01689006, 42.04954, 42.00081, 40.91390027, 40.8552717, 40.12104, 41.18901979, 41.39717327, 41.44383, 41.38588, 41.25379, 41.54796, 41.43915579, 41.13021, 41.54951, 41.72216, 40.62536, 41.44000545, 42.0110358, 42.03216628, 41.73747, 41.38021, 41.18216, 41.54929978, 41.61052, 41.61879, 41.13542, 41.47349391, 41.79156731],
    '2010':[41.64527427, 42.26671348, 41.35293206, 42.28311527, 41.21427866, 42.16562854, 41.75354552, 41.40372087, 40.89389467, 40.71613, 41.72457066, 41.37818845, 40.45797, 40.96458, 40.97601, 41.25280152, 41.5288795, 41.12155, 43.142338, 41.58172978, 41.87959845, 41.42867, 41.71299462, 41.86886, 42.29183, 41.9588, 41.70714, 42.46185915, 42.80663, 41.89799477, 42.36942, 42.06153, 42.2629111, 42.43, 42.18062, 40.9575993, 40.8764358, 40.08676, 41.26710191, 41.32329215, 41.37772, 41.20456, 41.3038, 41.6097, 41.54179233, 41.42257, 41.65101, 41.73242, 40.58368, 41.54378458, 42.03434098, 42.05697961, 41.74125, 41.42074, 41.45257, 41.52160473, 41.51817, 41.74478, 41.01266, 41.49021468, 41.83558876],
    '2011':[41.69776018, 42.89966856, 41.40400845, 42.46550391, 41.2457176, 42.20912182, 41.78431167, 41.47556954, 40.96413653, 40.69048, 41.7889029, 41.3702853, 40.49022, 41.07943, 41.00738, 41.22187635, 41.50208462, 41.17578, 43.12630691, 41.67751974, 42.04682943, 41.48775, 41.73131277, 41.91608803, 42.39085, 42.01629, 41.73503, 42.38475862, 42.7891, 41.97986519, 42.07914, 42.04469, 42.33941069, 42.51858, 42.25117, 40.99544944, 40.95049086, 40.25759, 41.24728927, 41.45979953, 41.4721, 41.51726, 41.33739, 41.74741, 41.49713145, 41.28987, 41.54871, 41.73417, 40.57541, 41.59210915, 42.12162266, 42.14422975, 41.82894, 41.31664, 41.43991, 41.49151356, 41.48566, 41.71249, 40.99764, 41.57890702, 41.96510391],
    '2012':[41.69053485, 42.96083812, 41.36277887, 42.22727921, 41.25125413, 42.18618305, 41.7805152, 41.406524, 41.02913146, 40.71132, 41.65334095, 41.43352659, 40.48532, 41.04482, 41.09306, 41.22623244, 41.51652711, 41.1073, 43.15281169, 41.5338158, 41.83656144, 41.37825, 41.73219672, 41.90346309, 42.30641, 41.94429, 41.77713, 42.3358687, 42.74475, 41.88706289, 41.99267, 42.0336, 42.37419352, 42.54813, 42.28853, 40.95532936, 40.83562466, 40.36852, 41.19228376, 41.52042114, 41.54647, 41.52173, 41.42971, 41.48821, 41.45080075, 41.3476, 41.53607, 41.57648, 40.55928, 41.57434092, 42.09516519, 42.12618478, 41.69357, 41.4421, 41.42191, 41.51967212, 41.47171, 41.75935, 41.15463, 41.64868836, 41.94273083],
    '2013':[41.68490733, 42.83769643, 41.37596981, 42.20569453, 41.26928905, 42.23377333, 41.68148832, 41.49788912, 40.96782179, 40.70785, 41.66985913, 41.3934543, 40.48848, 41.12758, 41.01208, 41.21027276, 41.40375393, 41.18986, 43.12249424, 41.51523927, 41.79964604, 41.37531, 41.72306724, 41.91165898, 42.31969, 41.94519, 41.77869, 42.39570192, 42.82637, 41.86375052, 41.95046, 42.05049, 42.37382049, 42.56189, 42.28896, 40.96523043, 40.84946458, 40.3802, 41.17871327, 41.50111329, 41.50732, 41.48373, 41.50339, 41.47857, 41.45255092, 41.4175, 41.39097, 41.62007, 40.81326, 41.51930487, 41.9743506, 42.14210548, 41.67963, 41.41989, 41.48074, 41.52428547, 41.52152, 41.69666, 41.21751, 41.7105608, 41.9323978],
    '2014':[41.6663703, 42.75956958, 41.35906942, 42.19954349, 41.26342196, 42.25948186, 41.61439186, 41.56737214, 40.99346726, 40.5975, 41.66983903, 41.29894172, 40.48759, 40.93655, 40.9848, 41.60016231, 41.50408554, 41.24193, 43.1041709, 41.46204989, 41.75161747, 41.31976, 41.70805459, 41.90764602, 42.30084, 41.95448, 41.76935, 42.4361608, 42.86273, 41.89939512, 42.01961, 42.06413, 42.38816168, 42.60589, 42.29052, 41.03519144, 41.09112593, 40.39584, 41.1852993, 41.4181976, 41.46447, 41.25514, 41.44814, 41.50111, 41.41697183, 41.39638, 41.22514, 41.67398, 40.78824, 41.49797139, 41.96935861, 42.14653443, 41.62734, 41.4692, 41.39647, 41.5132754, 41.52322, 41.6203, 41.27965, 41.86763544, 41.80748591],
    '2015':[41.68274825, 42.84937942, 41.38989641, 42.35984052, 41.34190543, 42.2262608, 41.77358743, 41.66480612, 41.0087425, 40.57642, 41.73042626, 41.43181149, 40.52198, 41.2123, 41.2452, 41.63747396, 41.49219179, 41.21378, 43.04345032, 41.40159962, 41.68464903, 41.26393, 41.71745656, 41.89185698, 42.27951, 41.95074, 41.74314, 42.42012027, 42.83703, 42.00468581, 41.97995, 42.04265, 42.38526517, 42.60845, 42.2837, 41.07465012, 40.99176143, 40.4006, 41.28586069, 41.4823194, 41.47175, 41.45319, 41.53115, 41.55262, 41.46020921, 41.42041, 41.25984, 41.68531, 40.75773, 41.64916399, 41.91791644, 42.10765468, 41.57833, 41.48844, 41.39847, 41.52239003, 41.49764, 41.68459, 41.3228, 41.86658854, 41.90999984],
    '2016':[41.70140577, 42.82512215, 41.38104558, 42.42092839, 41.33589588, 42.25411637, 41.66090667, 41.72980951, 41.41476073, 40.43776, 41.74513182, 41.45705564, 40.54016, 41.04577, 41.13931, 41.13509682, 41.53212526, 41.2241, 42.95062506, 41.38626409, 41.60713736, 41.27881, 41.7451025, 41.93778311, 42.3224, 42.03921, 41.75822, 42.41897643, 42.90678, 41.92707325, 41.83163, 42.05202, 42.4072359, 42.58243, 42.32949, 41.04498874, 40.94625522, 40.378, 41.2614234, 41.53955804, 41.59361, 41.44827, 41.50287, 41.47387, 41.52555532, 41.45892, 41.43955, 41.71098, 40.69638, 41.69488717, 41.92120066, 42.11806923, 41.56484, 41.4621, 41.43441, 41.56580389, 41.59293, 41.70802, 41.22221, 41.85877037, 41.97840448],
    '2017':[41.69132722, 42.6297354, 41.34594185, 42.65426811, 41.29957947, 42.22939937, 41.61775777, 41.73714492, 41.31357519, 40.43222, 41.68560916, 41.43153271, 40.47137, 40.93648, 41.08315, 41.02908706, 41.57502587, 41.18677, 42.88530719, 41.34928241, 41.60589059, 41.22811, 41.74893612, 41.92609063, 42.29972, 41.78863, 41.92237, 42.43528723, 42.9136, 41.89711989, 41.87796, 42.06118, 42.45601154, 42.64666, 42.37238, 41.03778697, 40.92641168, 40.34976, 41.27900409, 41.50871084, 41.59633, 41.22966, 41.55927, 41.66534, 41.52912025, 41.56459, 41.43512, 41.68702, 40.70344, 41.63141615, 41.93436578, 42.12748004, 41.5653, 41.46786, 41.40524, 41.56839559, 41.57768, 41.71406, 41.28935, 41.9360592, 42.02345542],
    '2018':[41.68553783, 42.8515483, 41.33811913, 42.65758609, 41.2875942, 42.20643119, 41.63364702, 41.77257419, 41.25679209, 40.44605, 41.69622949, 41.36571116, 40.47147, 40.9493, 41.05453, 41.01822916, 41.552874, 41.17039, 43.03690096, 41.33945195, 41.61383552, 41.21076, 41.73298639, 41.92819522, 42.27618, 42.02829, 41.75682, 42.41374415, 42.78443, 41.99285193, 42.03988, 42.07963, 42.42671068, 42.65618, 42.32718, 41.04324224, 40.88494136, 40.41251, 41.27514214, 41.50936139, 41.63876, 41.25306, 41.45626, 41.61317, 41.49363604, 41.46864, 41.24419, 41.69345, 40.82295, 41.7910669, 41.93681474, 42.15972544, 41.52786, 41.50461, 41.38553, 41.57014069, 41.55823, 41.74799, 41.31577, 41.9144178, 41.84379439],
    '2019':[41.68590754, 43.14416623, 41.30942694, 42.46946429, 41.27455855, 42.17914124, 41.62011472, 41.74069857, 41.30902418, 40.41444, 41.63607716, 41.39580043, 40.47896, 40.84519, 41.01408, 40.93153796, 41.60325541, 41.14801, 42.95545753, 41.28355938, 41.55912336, 41.15503, 41.72870325, 41.90454888, 42.27214, 41.9904, 41.73755, 42.39516937, 42.73916, 41.9657957, 42.0164, 42.1172, 42.42633768, 42.62561, 42.34071, 41.00056323, 40.85432958, 40.46726, 41.18867751, 41.49172222, 41.5467, 41.32239, 41.52488, 41.67187, 41.54734902, 41.55473, 41.47582, 41.68928, 40.79317, 41.67800523, 41.86487553, 42.07568675, 41.49836, 41.58437, 41.44241, 41.57458722, 41.56957, 41.78089, 41.25294, 41.72248321, 41.82114729],
    '2020':[41.69615701, 43.22707611, 41.30476877, 42.39258197, 41.27783553, 42.19575892, 41.70851974, 41.74911355, 41.20317035, 40.41176, 41.64654117, 41.36311939, 40.56528, 40.82677, 41.00998, 41.03844379, 41.5926928, 41.14671, 42.90501929, 41.2662673, 41.55289773, 41.13372, 41.74119534, 41.91149512, 42.26695, 42.01536, 41.73402, 42.41194036, 42.72073, 42.02163835, 42.11782, 42.09892, 42.48657151, 42.72619, 42.38563, 41.04897761, 40.93076919, 40.48398, 41.22359748, 41.51590532, 41.52673, 41.40041, 41.57786, 41.58269, 41.62513955, 41.52213, 41.70097, 41.69473, 40.97845, 41.73993595, 41.83806234, 42.03756648, 41.49934, 41.62244, 41.46966, 41.54949873, 41.5525, 41.70301, 41.29011, 41.68737301, 41.82666124],
    '2021':[41.70402195, 43.28493039, 41.31154782, 42.38986581, 41.28408805, 42.19454862, 41.80572464, 41.76110806, 41.21268567, 40.52297, 41.66247719, 41.32522455, 40.54261, 40.87308, 41.0824, 41.00561578, 41.51782658, 41.22306, 42.91486025, 41.26499732, 41.53796329, 41.13965, 41.74747679, 41.92327681, 42.29163, 42.03995, 41.73122, 42.41936288, 42.71537, 41.96749494, 42.17934, 42.08287, 42.51747689, 42.77873, 42.40817, 41.10201153, 40.98511513, 40.60808, 41.24858365, 41.52632806, 41.56223, 41.46349, 41.51121, 41.59693, 41.63059679, 41.40153, 41.73769, 41.73132, 41.05249, 41.75957991, 41.84565001, 42.05931912, 41.47628, 41.57416, 41.40797, 41.58963454, 41.58209, 41.81614, 41.25979, 41.61147361, 41.88018543],
    '2022':[41.6944123, 43.38451548, 41.28927145, 42.43141271, 41.25644706, 42.20048003, 41.84034254, 41.78572639, 41.12940596, 40.49089, 41.60345163, 41.30745711, 40.49015, 40.85936, 41.02153, 40.96536623, 41.52319905, 41.20432, 43.03887566, 41.24171165, 41.51164344, 41.11881, 41.73590552, 41.89933543, 42.24492, 42.02191, 41.71249, 42.43128224, 42.74728, 42.01492855, 42.13897, 42.08058, 42.48741023, 42.73192, 42.38913, 41.121261, 40.97150658, 40.57926, 41.27960296, 41.58486059, 41.63944, 41.41605, 41.61286, 41.62238, 41.60694379, 41.45849, 41.65889, 41.70078, 40.7896, 41.87456906, 41.8452464, 42.04338503, 41.48359, 41.55934, 41.44283, 41.57474293, 41.57821, 41.75435, 41.27899, 41.66769488, 41.82278643],
    '2023':[41.6944123, 43.38451548, 41.28927145, 42.43141271, 41.25644706, 42.20048003, 41.84034254, 41.78572639, 41.12940596, 40.49089, 41.60345163, 41.30745711, 40.49015, 40.85936, 41.02153, 40.96536623, 41.52319905, 41.20432, 43.03887566, 41.24171165, 41.51164344, 41.11881, 41.73590552, 41.89933543, 42.24492, 42.02191, 41.71249, 42.43128224, 42.74728, 42.01492855, 42.13897, 42.08058, 42.48741023, 42.73192, 42.38913, 41.121261, 40.97150658, 40.57926, 41.27960296, 41.58486059, 41.63944, 41.41605, 41.61286, 41.62238, 41.60694379, 41.45849, 41.65889, 41.70078, 40.7896, 41.87456906, 41.8452464, 42.04338503, 41.48359, 41.55934, 41.44283, 41.57474293, 41.57821, 41.75435, 41.27899, 41.66769488, 41.82278643],
    '2024':[41.6944123, 43.38451548, 41.28927145, 42.43141271, 41.25644706, 42.20048003, 41.84034254, 41.78572639, 41.12940596, 40.49089, 41.60345163, 41.30745711, 40.49015, 40.85936, 41.02153, 40.96536623, 41.52319905, 41.20432, 43.03887566, 41.24171165, 41.51164344, 41.11881, 41.73590552, 41.89933543, 42.24492, 42.02191, 41.71249, 42.43128224, 42.74728, 42.01492855, 42.13897, 42.08058, 42.48741023, 42.73192, 42.38913, 41.121261, 40.97150658, 40.57926, 41.27960296, 41.58486059, 41.63944, 41.41605, 41.61286, 41.62238, 41.60694379, 41.45849, 41.65889, 41.70078, 40.7896, 41.87456906, 41.8452464, 42.04338503, 41.48359, 41.55934, 41.44283, 41.57474293, 41.57821, 41.75435, 41.27899, 41.66769488, 41.82278643],
    '2025':[41.6944123, 43.38451548, 41.28927145, 42.43141271, 41.25644706, 42.20048003, 41.84034254, 41.78572639, 41.12940596, 40.49089, 41.60345163, 41.30745711, 40.49015, 40.85936, 41.02153, 40.96536623, 41.52319905, 41.20432, 43.03887566, 41.24171165, 41.51164344, 41.11881, 41.73590552, 41.89933543, 42.24492, 42.02191, 41.71249, 42.43128224, 42.74728, 42.01492855, 42.13897, 42.08058, 42.48741023, 42.73192, 42.38913, 41.121261, 40.97150658, 40.57926, 41.27960296, 41.58486059, 41.63944, 41.41605, 41.61286, 41.62238, 41.60694379, 41.45849, 41.65889, 41.70078, 40.7896, 41.87456906, 41.8452464, 42.04338503, 41.48359, 41.55934, 41.44283, 41.57474293, 41.57821, 41.75435, 41.27899, 41.66769488, 41.82278643],
    '2026':[41.6944123, 43.38451548, 41.28927145, 42.43141271, 41.25644706, 42.20048003, 41.84034254, 41.78572639, 41.12940596, 40.49089, 41.60345163, 41.30745711, 40.49015, 40.85936, 41.02153, 40.96536623, 41.52319905, 41.20432, 43.03887566, 41.24171165, 41.51164344, 41.11881, 41.73590552, 41.89933543, 42.24492, 42.02191, 41.71249, 42.43128224, 42.74728, 42.01492855, 42.13897, 42.08058, 42.48741023, 42.73192, 42.38913, 41.121261, 40.97150658, 40.57926, 41.27960296, 41.58486059, 41.63944, 41.41605, 41.61286, 41.62238, 41.60694379, 41.45849, 41.65889, 41.70078, 40.7896, 41.87456906, 41.8452464, 42.04338503, 41.48359, 41.55934, 41.44283, 41.57474293, 41.57821, 41.75435, 41.27899, 41.66769488, 41.82278643],
    '2027':[41.6944123, 43.38451548, 41.28927145, 42.43141271, 41.25644706, 42.20048003, 41.84034254, 41.78572639, 41.12940596, 40.49089, 41.60345163, 41.30745711, 40.49015, 40.85936, 41.02153, 40.96536623, 41.52319905, 41.20432, 43.03887566, 41.24171165, 41.51164344, 41.11881, 41.73590552, 41.89933543, 42.24492, 42.02191, 41.71249, 42.43128224, 42.74728, 42.01492855, 42.13897, 42.08058, 42.48741023, 42.73192, 42.38913, 41.121261, 40.97150658, 40.57926, 41.27960296, 41.58486059, 41.63944, 41.41605, 41.61286, 41.62238, 41.60694379, 41.45849, 41.65889, 41.70078, 40.7896, 41.87456906, 41.8452464, 42.04338503, 41.48359, 41.55934, 41.44283, 41.57474293, 41.57821, 41.75435, 41.27899, 41.66769488, 41.82278643],
    '2028':[41.6944123, 43.38451548, 41.28927145, 42.43141271, 41.25644706, 42.20048003, 41.84034254, 41.78572639, 41.12940596, 40.49089, 41.60345163, 41.30745711, 40.49015, 40.85936, 41.02153, 40.96536623, 41.52319905, 41.20432, 43.03887566, 41.24171165, 41.51164344, 41.11881, 41.73590552, 41.89933543, 42.24492, 42.02191, 41.71249, 42.43128224, 42.74728, 42.01492855, 42.13897, 42.08058, 42.48741023, 42.73192, 42.38913, 41.121261, 40.97150658, 40.57926, 41.27960296, 41.58486059, 41.63944, 41.41605, 41.61286, 41.62238, 41.60694379, 41.45849, 41.65889, 41.70078, 40.7896, 41.87456906, 41.8452464, 42.04338503, 41.48359, 41.55934, 41.44283, 41.57474293, 41.57821, 41.75435, 41.27899, 41.66769488, 41.82278643],
    '2029':[41.6944123, 43.38451548, 41.28927145, 42.43141271, 41.25644706, 42.20048003, 41.84034254, 41.78572639, 41.12940596, 40.49089, 41.60345163, 41.30745711, 40.49015, 40.85936, 41.02153, 40.96536623, 41.52319905, 41.20432, 43.03887566, 41.24171165, 41.51164344, 41.11881, 41.73590552, 41.89933543, 42.24492, 42.02191, 41.71249, 42.43128224, 42.74728, 42.01492855, 42.13897, 42.08058, 42.48741023, 42.73192, 42.38913, 41.121261, 40.97150658, 40.57926, 41.27960296, 41.58486059, 41.63944, 41.41605, 41.61286, 41.62238, 41.60694379, 41.45849, 41.65889, 41.70078, 40.7896, 41.87456906, 41.8452464, 42.04338503, 41.48359, 41.55934, 41.44283, 41.57474293, 41.57821, 41.75435, 41.27899, 41.66769488, 41.82278643],
    '2030':[41.6944123, 43.38451548, 41.28927145, 42.43141271, 41.25644706, 42.20048003, 41.84034254, 41.78572639, 41.12940596, 40.49089, 41.60345163, 41.30745711, 40.49015, 40.85936, 41.02153, 40.96536623, 41.52319905, 41.20432, 43.03887566, 41.24171165, 41.51164344, 41.11881, 41.73590552, 41.89933543, 42.24492, 42.02191, 41.71249, 42.43128224, 42.74728, 42.01492855, 42.13897, 42.08058, 42.48741023, 42.73192, 42.38913, 41.121261, 40.97150658, 40.57926, 41.27960296, 41.58486059, 41.63944, 41.41605, 41.61286, 41.62238, 41.60694379, 41.45849, 41.65889, 41.70078, 40.7896, 41.87456906, 41.8452464, 42.04338503, 41.48359, 41.55934, 41.44283, 41.57474293, 41.57821, 41.75435, 41.27899, 41.66769488, 41.82278643],
    '2031':[41.6944123, 43.38451548, 41.28927145, 42.43141271, 41.25644706, 42.20048003, 41.84034254, 41.78572639, 41.12940596, 40.49089, 41.60345163, 41.30745711, 40.49015, 40.85936, 41.02153, 40.96536623, 41.52319905, 41.20432, 43.03887566, 41.24171165, 41.51164344, 41.11881, 41.73590552, 41.89933543, 42.24492, 42.02191, 41.71249, 42.43128224, 42.74728, 42.01492855, 42.13897, 42.08058, 42.48741023, 42.73192, 42.38913, 41.121261, 40.97150658, 40.57926, 41.27960296, 41.58486059, 41.63944, 41.41605, 41.61286, 41.62238, 41.60694379, 41.45849, 41.65889, 41.70078, 40.7896, 41.87456906, 41.8452464, 42.04338503, 41.48359, 41.55934, 41.44283, 41.57474293, 41.57821, 41.75435, 41.27899, 41.66769488, 41.82278643],
    '2032':[41.6944123, 43.38451548, 41.28927145, 42.43141271, 41.25644706, 42.20048003, 41.84034254, 41.78572639, 41.12940596, 40.49089, 41.60345163, 41.30745711, 40.49015, 40.85936, 41.02153, 40.96536623, 41.52319905, 41.20432, 43.03887566, 41.24171165, 41.51164344, 41.11881, 41.73590552, 41.89933543, 42.24492, 42.02191, 41.71249, 42.43128224, 42.74728, 42.01492855, 42.13897, 42.08058, 42.48741023, 42.73192, 42.38913, 41.121261, 40.97150658, 40.57926, 41.27960296, 41.58486059, 41.63944, 41.41605, 41.61286, 41.62238, 41.60694379, 41.45849, 41.65889, 41.70078, 40.7896, 41.87456906, 41.8452464, 42.04338503, 41.48359, 41.55934, 41.44283, 41.57474293, 41.57821, 41.75435, 41.27899, 41.66769488, 41.82278643],
    '2033':[41.6944123, 43.38451548, 41.28927145, 42.43141271, 41.25644706, 42.20048003, 41.84034254, 41.78572639, 41.12940596, 40.49089, 41.60345163, 41.30745711, 40.49015, 40.85936, 41.02153, 40.96536623, 41.52319905, 41.20432, 43.03887566, 41.24171165, 41.51164344, 41.11881, 41.73590552, 41.89933543, 42.24492, 42.02191, 41.71249, 42.43128224, 42.74728, 42.01492855, 42.13897, 42.08058, 42.48741023, 42.73192, 42.38913, 41.121261, 40.97150658, 40.57926, 41.27960296, 41.58486059, 41.63944, 41.41605, 41.61286, 41.62238, 41.60694379, 41.45849, 41.65889, 41.70078, 40.7896, 41.87456906, 41.8452464, 42.04338503, 41.48359, 41.55934, 41.44283, 41.57474293, 41.57821, 41.75435, 41.27899, 41.66769488, 41.82278643],
    '2034':[41.6944123, 43.38451548, 41.28927145, 42.43141271, 41.25644706, 42.20048003, 41.84034254, 41.78572639, 41.12940596, 40.49089, 41.60345163, 41.30745711, 40.49015, 40.85936, 41.02153, 40.96536623, 41.52319905, 41.20432, 43.03887566, 41.24171165, 41.51164344, 41.11881, 41.73590552, 41.89933543, 42.24492, 42.02191, 41.71249, 42.43128224, 42.74728, 42.01492855, 42.13897, 42.08058, 42.48741023, 42.73192, 42.38913, 41.121261, 40.97150658, 40.57926, 41.27960296, 41.58486059, 41.63944, 41.41605, 41.61286, 41.62238, 41.60694379, 41.45849, 41.65889, 41.70078, 40.7896, 41.87456906, 41.8452464, 42.04338503, 41.48359, 41.55934, 41.44283, 41.57474293, 41.57821, 41.75435, 41.27899, 41.66769488, 41.82278643],
    '2035':[41.6944123, 43.38451548, 41.28927145, 42.43141271, 41.25644706, 42.20048003, 41.84034254, 41.78572639, 41.12940596, 40.49089, 41.60345163, 41.30745711, 40.49015, 40.85936, 41.02153, 40.96536623, 41.52319905, 41.20432, 43.03887566, 41.24171165, 41.51164344, 41.11881, 41.73590552, 41.89933543, 42.24492, 42.02191, 41.71249, 42.43128224, 42.74728, 42.01492855, 42.13897, 42.08058, 42.48741023, 42.73192, 42.38913, 41.121261, 40.97150658, 40.57926, 41.27960296, 41.58486059, 41.63944, 41.41605, 41.61286, 41.62238, 41.60694379, 41.45849, 41.65889, 41.70078, 40.7896, 41.87456906, 41.8452464, 42.04338503, 41.48359, 41.55934, 41.44283, 41.57474293, 41.57821, 41.75435, 41.27899, 41.66769488, 41.82278643],
    '2036':[41.6944123, 43.38451548, 41.28927145, 42.43141271, 41.25644706, 42.20048003, 41.84034254, 41.78572639, 41.12940596, 40.49089, 41.60345163, 41.30745711, 40.49015, 40.85936, 41.02153, 40.96536623, 41.52319905, 41.20432, 43.03887566, 41.24171165, 41.51164344, 41.11881, 41.73590552, 41.89933543, 42.24492, 42.02191, 41.71249, 42.43128224, 42.74728, 42.01492855, 42.13897, 42.08058, 42.48741023, 42.73192, 42.38913, 41.121261, 40.97150658, 40.57926, 41.27960296, 41.58486059, 41.63944, 41.41605, 41.61286, 41.62238, 41.60694379, 41.45849, 41.65889, 41.70078, 40.7896, 41.87456906, 41.8452464, 42.04338503, 41.48359, 41.55934, 41.44283, 41.57474293, 41.57821, 41.75435, 41.27899, 41.66769488, 41.82278643],
    '2037':[41.6944123, 43.38451548, 41.28927145, 42.43141271, 41.25644706, 42.20048003, 41.84034254, 41.78572639, 41.12940596, 40.49089, 41.60345163, 41.30745711, 40.49015, 40.85936, 41.02153, 40.96536623, 41.52319905, 41.20432, 43.03887566, 41.24171165, 41.51164344, 41.11881, 41.73590552, 41.89933543, 42.24492, 42.02191, 41.71249, 42.43128224, 42.74728, 42.01492855, 42.13897, 42.08058, 42.48741023, 42.73192, 42.38913, 41.121261, 40.97150658, 40.57926, 41.27960296, 41.58486059, 41.63944, 41.41605, 41.61286, 41.62238, 41.60694379, 41.45849, 41.65889, 41.70078, 40.7896, 41.87456906, 41.8452464, 42.04338503, 41.48359, 41.55934, 41.44283, 41.57474293, 41.57821, 41.75435, 41.27899, 41.66769488, 41.82278643]
}
th = pd.DataFrame(data)

In [119]:
th_ids = data['id']
labels = {}
t1s = {}
for i,l in enumerate(data['id']):
    labels[l] = data['label'][i]
    t1s[l] = t1[i]
labels

{'01-96': 'TOTAL',
 '01-03': 'SECTEUR PRIMAIRE',
 '05-43': 'SECTEUR SECONDAIRE',
 '5-9': 'Industries extractives',
 '10-33': 'Industrie manufacturière',
 '10-12': 'Industries alimentaires et du tabac',
 '13-15': 'Industries du textile et de l’habillement',
 '16-18': 'Industries du bois et du papier\xa0; imprimerie',
 '19-20': 'Cokéfaction, raffinage et industrie chimique',
 '21': 'Industrie pharmaceutique',
 '22-23': 'Industries du caoutchouc et du plastique',
 '24-25': 'Fabrication de produits métalliques',
 '26': 'Fabrication de produits électroniques; horlogerie',
 '27': 'Fabrication d’équipements électriques',
 '28': 'Fabrication de machines et équipements n.c.a',
 '29-30': 'Fabrication de matériels de transport',
 '31-33': 'Autres industries manufacturières; rép. et inst.',
 '35': 'Production et distribution d’énergie',
 '36-39': 'Production et distr. d’eau; gestion des déchets',
 '41-43': 'Construction',
 '41-42': 'Construction de bâtiments et génie civil',
 '43': 'Travaux de con

## Data processing

In [120]:
t1 = {
    'homme': t1_1_10,
    'femme': t1_2_10,
    '26 al. 6 RAI': t1_10
}

In [121]:
def to_range(id):
    if '+' in id:
        id_range = [77, 79, 80, 81, 82]
    elif '/' in id:
        if '-' in id:
            ranges = [i for i in id.split('/')]
            print(ranges)
            id_range = []
            for r in ranges:
                print(r)
                rr = [int(i) for i in r.split('-')]
                print(rr)
                id_range.extend(i for i in range(rr[0], rr[-1]+1))
        else:
            id_range = [int(i) for i in id.split('/')]
    elif '-' in id:
        id_range = [int(i) for i in id.split('-')]
        id_range = [i for i in range(id_range[0], id_range[-1]+1)]
        # print(id_range)
    else:
        id_range = [int(id)]

    # print(id_range)
    return id_range

In [122]:
th_index = []
t1_rai_index = []
t1_hf_index = []
for id in th_ids:
    th_index.append({"id": id, "index": to_range(id)})
for id in t1_rai_ids:
    t1_rai_index.append({"id": id, "index": to_range(id)})
for id in t1_hf_ids:
    t1_hf_index.append({"id": id, "index": to_range(id)})

['05-09', '35-39']
05-09
[5, 9]
35-39
[35, 39]


In [123]:
t1_index = {
    'homme': t1_hf_index,
    'femme': t1_hf_index,
    '26 al. 6 RAI': t1_rai_index
}

In [187]:
def get_id(id, index):
    res = []
    for i in index:
        is_in = True
        for r in to_range(id):
            if r not in i['index']:
                is_in = False
                break
        if is_in:
            res.append(i)
    if len(res) == 0:
        print(res[0]['id'])
        return res[0]['id']
    elif len(res) > 1:
        final = res[0]
        for r in res:
            if len(final['index']) > len(r['index']):
                final = r
        # print(final['id'])
        return final['id']
    else:
        print('id not in index')
        return None
get_id(beneficiary['branche'], t1_index[beneficiary["sexe"]])

id not in index


## Fonction de calcul

In [125]:
def calcul_salaire(beneficiary, type_salaire):
    year = type_salaire['année']
    salaire = type_salaire['salaire']
    if salaire not in [False, None, ''] and year not in [False, None, '']:
        indexation_annees = []
        base_index = get_indexation_t39(beneficiary, year)
        print("Indexation T1:")
        print(f"Année: {year}, Indice: {base_index}")

        year_index = get_indexation_t39(beneficiary, beneficiary['ess'])
        indexation_annees.append({
            'année': beneficiary['ess'],
            'indice': year_index,
            'revenu': salaire / base_index * year_index
        })
        print(f"Année: {indexation_annees[-1]['année']}, Indice: {round(indexation_annees[-1]['indice'], 2)}, Revenu: {round(indexation_annees[-1]['revenu'], 2)} CHF\n")
        return indexation_annees
    else:
        return [{"revenu": 0}]

In [190]:
def calc_salaire_effectif(beneficiary):
    salef = beneficiary['salaire_effectif']
    if salef['année'] == '' or salef['salaire'] == '':
        res = [{"revenu": 0}]
        print(f"Salaire effectif à prendre en compte? Non\n")
    else:
        print(f"Salaire effectif à prendre en compte? Oui: année: {salef['année']}, salaire: {round(salef['salaire'], 2)} CHF")
        # print(f"Branche économique selon TA1 N° {beneficiary['branche']}   {t1s[beneficiary['branche']]}\nLibellé: {labels[beneficiary['branche']]}")
        res = calcul_salaire(beneficiary, salef)
    return res

In [168]:
# Function to calculate indexation_annees
def calculate_indexation(beneficiary, salaire, year):
    salaire_annuel = calc_salaire_annuel(salaire)
    # print(f"Branche économique selon TA1 N° {beneficiary['branche']}   \nLibellé: {labels[beneficiary['branche']]}\nNiveau de compétence selon TA1: {beneficiary['niveau_comp']}\n Salaire OFS de la branche (TA1): année: {beneficiary['salaire_ofs']['année']}, salaire: {round(beneficiary['salaire_ofs']['salaire'], 2)} CHF")
    print((f"Salaire OFS sur xx heures hebdomdaire x 12: {round(salaire_annuel['heures_normales'], 2)}, {round(salaire_annuel['salaire_normal'], 2)} CHF, {round(salaire_annuel['salaire_normal_annuel'], 2)} CHF"))
    if salaire_annuel['salaire_normal_annuel'] not in [False, None, '']:
        indexation_annees = []
        base_index = get_indexation_t39(beneficiary, year)
        print("Indexation T1:")
        print(f"Année: {year}, Indice: {base_index}")

        year_index = get_indexation_t39(beneficiary, beneficiary['ess'])
        indexation_annees.append({
            'année': beneficiary['ess'],
            'indice': year_index,
            'revenu': salaire_annuel['salaire_normal_annuel'] / base_index * year_index
        })
        print(f"Année: {round(indexation_annees[-1]['année'], 2)}, Indice: {round(indexation_annees[-1]['indice'], 2)}, Revenu: {round(indexation_annees[-1]['revenu'], 2)} CHF\n")
        return indexation_annees
    else:
        return None

In [196]:
### Fonction de calcul
def calc_salaire_as(beneficiary):
    salas = beneficiary['salaire_as']
    if salas['année'] == '' or salas['salaire'] == '':
        res = [{"revenu": 0}]
        # print(f"Salaire effectif à prendre en compte? Non\n")
    else:
        print(
            f"RS:\nSalaire avant atteinte à la santé: année: {salas['année']}, salaire: {round(salas['salaire'], 2)} CHF")
        # print(f"Branche économique selon TA1 N° {beneficiary['branche']}   nLibellé: {labels[beneficiary['branche']]}")
        res = calcul_salaire(beneficiary, salas)
    return res

In [129]:
def indexation_t39(beneficiary, evo_salaires):
    # Get the value of C1, which is beneficiary["ess"]
    c1 = beneficiary["ess"]

    # Get the maximum value from the "année" column in evo_salaires
    max_annee = evo_salaires["année"].max()

    # Apply the logic based on the Excel formula
    if c1 == "année":
        return ""
    elif c1 > max_annee:
        return max_annee
    else:
        return c1

In [188]:
def get_indexation_t39(beneficiary, year):
    # Extract parameters
    sexe = beneficiary["sexe"]
    df = t1[sexe]
    branche = get_id(beneficiary["branche"], t1_index[sexe]) if beneficiary["branche"] != '05-96' else '05-96'
    print(branche)
    row_index = df[df['id'] == branche].index[0]

    column_index = df.columns.get_loc(f"{year}")
    return df.iloc[row_index, column_index]


In [163]:
def heures_hebdo(beneficiary, th):
    # Extract parameters
    year = beneficiary["ess"]
    branche = get_id(beneficiary["branche"], th_index) if beneficiary["branche"] != '05-96' else '01-96'

    # Find the matching row and column for the INDEX function
    row_index = th[th['id'] == branche].index[0]

    # Find the column index based on the header matching C11 (index_t39)
    column_index = th.columns.get_loc(f"{year}")

    # Use row_index and column_index to retrieve the value
    return th.iloc[row_index, column_index]

In [132]:
def calc_salaire_annuel(salaire):
    heures_normales = heures_hebdo(beneficiary, th)
    salaire_normal = salaire * heures_normales / 40
    salaire_normal_annuel = salaire_normal * 12
    return {'heures_normales': heures_normales, "salaire": salaire, "salaire_normal": salaire_normal, "salaire_normal_annuel": salaire_normal_annuel}

In [133]:
def calc_deduction(beneficiary, salaire):
    annee = int(beneficiary['ess'])
    d31 = beneficiary['horaire']/100
    d32 = beneficiary['diminution']/100
    if annee < 2024 and (d31 - (d31 * d32)) <= 0.5:
        deduction = 0.1
    elif annee < 2024 and (d31 - (d31 * d32)) > 0.5:
        deduction = 0
    elif annee >= 2024 and (d31 - (d31 * d32)) <= 0.5:
        deduction = 0.2
    else:
        deduction = 0.1

    deducted = salaire - (deduction * salaire)

    print(f"Déduction forfaitaire si CTAA <=50%: {deduction*100}%, {round(deducted, 2)} CHF")
    return  deducted

In [134]:
def calc_salaire_exigible(beneficiary, indexation):
    d34 = beneficiary['abattement']
    salaire_base = max([i['revenu'] for i in indexation]) * beneficiary['horaire'] / 100
    print(f"Horaire et pourcentage: {beneficiary['horaire']}%, {round(salaire_base, 2)} CHF")

    diminution_rendement = salaire_base - salaire_base * beneficiary['diminution'] / 100
    print(f"Diminution de rendement: {beneficiary['diminution']}%, {round(diminution_rendement, 2)} CHF")

    exigible = calc_deduction(beneficiary, diminution_rendement)

    if d34 not in [False, None, '', 0] and int(beneficiary['ess'] < 2024):
        exigible -= exigible * d34/100
        print(f"Prise en compte d'abattements suppl.: oui, {d34}%, {round(exigible, 2)} CHF")

    return exigible

In [135]:
def calc_ri(beneficiary, salef, salaire_exigible):

    # si ess < 2024 et max(salef)<
    c1 = beneficiary['ess']

    max_salef = max([i['revenu'] for i in salef])

    if (c1 < 2024 or c1 > 2023) and max_salef < salaire_exigible:
        return salaire_exigible
    else:
        return max_salef

## Fonction d'execution

#### RS

##### Double ESS

In [136]:
def exec_rs(beneficiary):
    print(f"ESS: {beneficiary['ess']}\nType de calcul: Double ESS\nSexe: {beneficiary['sexe']}\n\nRS:")
    indexation = calculate_indexation(beneficiary, beneficiary['salaire_ofs']['salaire'], beneficiary['salaire_ofs']['année'])
    return indexation

##### Mise en parallèle des revenus

In [137]:
def exec_rs_as(beneficiary):
    print(f"ESS: {beneficiary['ess']}\nType de calcul: Mise en parallèle des revenus\nSexe: {beneficiary['sexe']}\n")
    rs = calc_salaire_as(beneficiary)
    return rs

#### Parallélisation des revenus

In [138]:
def exec_parallélisation(beneficiary):
    print(f"Parallélisation des revenus:")
    print((f"Niveau de compétence selon TA1: {beneficiary['niveau_comp']}"))
    indexation = calculate_indexation(beneficiary, beneficiary['salaire_ofs']['salaire'], beneficiary['salaire_ofs']['année'])

    mpr = indexation[0]['revenu'] * 0.95
    print(f"\nMise en parallèle des revenus (95% de l'ESS): {round(mpr, 2)} CHF\n")
    # return {'indexation': indexation, 'mpr': mpr}
    return indexation,mpr

#### RI

##### Salaire effectif

In [139]:
def exec_salef(beneficiary):
    print('RI:')
    salef = calc_salaire_effectif(beneficiary)
    return salef

##### Salaire exigible

In [140]:
def exec_salex(beneficiary):
    print("RI selon base ESS:")
    salex = calc_salaire_exigible(beneficiary, calculate_indexation(beneficiary, beneficiary['salaire_ofs']['salaire'], beneficiary['salaire_ofs']['année']))
    return salex

#### Préjudice économique

##### Double ESS

In [192]:
def double_ess(beneficiary):
    indexation = exec_rs(beneficiary)
    salef = exec_salef(beneficiary)
    salex = exec_salex(beneficiary)
    print("\nPréjudice économique:")
    rs = max([i['revenu'] for i in indexation])
    print(f"Revenu sans invalidité RS: {round(rs, 2)} CHF")

    ri = calc_ri(beneficiary, salef, salex)
    print(f"Salaire exigible RI: {round(ri, 2)} CHF")

    prejudice = rs-ri
    print(f"Préjudice économique: {round(prejudice, 2)} CHF")

    prejudice_pourcentage = prejudice / rs * 100
    print(f"Préjudice économique en %: {round(prejudice_pourcentage, 2)}%")
    return round(prejudice_pourcentage, 2)

##### Mise en parallèle des revenus

In [142]:
def mise_en_parallele_des_revenus(beneficiary):
    rs = exec_rs_as(beneficiary)
    indexation, mpr = exec_parallélisation(beneficiary)
    salef = exec_salef(beneficiary)
    salex = exec_salex(beneficiary)
    print("\nPréjudice économique:")
    rs = rs[0]['revenu'] if rs[0]['revenu']>mpr else mpr
    print(rs)
    print(f"Revenu sans invalidité RS: {round(rs, 2)} CHF")

    ri = calc_ri(beneficiary, salef, salex)
    print(f"Salaire exigible RI: {round(ri, 2)} CHF")

    prejudice = rs-ri
    print(f"Préjudice économique: {round(prejudice, 2)} CHF")

    prejudice_pourcentage = prejudice / rs * 100
    print(f"Préjudice économique en %: {round(prejudice_pourcentage, 2)}%")

## Calcul

### Double ESS

In [194]:
double_ess(beneficiary)

ESS: 2022
Type de calcul: Double ESS
Sexe: homme

RS:
Salaire OFS sur xx heures hebdomdaire x 12: 41.69, 5529.72 CHF, 66356.66 CHF
05-96
Indexation T1:
Année: 2022, Indice: 107.1
05-96
Année: 2022, Indice: 107.1, Revenu: 66356.66 CHF

RI:
Salaire effectif à prendre en compte? Oui: année: 0, salaire: 0 CHF
RI selon base ESS:
Salaire OFS sur xx heures hebdomdaire x 12: 41.69, 5529.72 CHF, 66356.66 CHF
05-96
Indexation T1:
Année: 2022, Indice: 107.1
05-96
Année: 2022, Indice: 107.1, Revenu: 66356.66 CHF

Horaire et pourcentage: 100%, 66356.66 CHF
Diminution de rendement: 50%, 33178.33 CHF
Déduction forfaitaire si CTAA <=50%: 10.0%, 29860.5 CHF

Préjudice économique:
Revenu sans invalidité RS: 66356.66 CHF
Salaire exigible RI: 29860.5 CHF
Préjudice économique: 36496.16 CHF
Préjudice économique en %: 55.0%


55.0

In [193]:
get_id(beneficiary["branche"], t1_index['homme'])

id not in index


### Mise en parallèle des revenus

In [197]:
mise_en_parallele_des_revenus(beneficiary)

ESS: 2022
Type de calcul: Mise en parallèle des revenus
Sexe: homme

RS:
Salaire avant atteinte à la santé: année: 2021, salaire: 90000 CHF
05-96
Indexation T1:
Année: 2021, Indice: 106.0
05-96
Année: 2022, Indice: 107.1, Revenu: 90933.96 CHF

Parallélisation des revenus:
Niveau de compétence selon TA1: 1
Salaire OFS sur xx heures hebdomdaire x 12: 41.69, 5529.72 CHF, 66356.66 CHF
05-96
Indexation T1:
Année: 2022, Indice: 107.1
05-96
Année: 2022, Indice: 107.1, Revenu: 66356.66 CHF


Mise en parallèle des revenus (95% de l'ESS): 63038.82 CHF

RI:
Salaire effectif à prendre en compte? Oui: année: 0, salaire: 0 CHF
RI selon base ESS:
Salaire OFS sur xx heures hebdomdaire x 12: 41.69, 5529.72 CHF, 66356.66 CHF
05-96
Indexation T1:
Année: 2022, Indice: 107.1
05-96
Année: 2022, Indice: 107.1, Revenu: 66356.66 CHF

Horaire et pourcentage: 100%, 66356.66 CHF
Diminution de rendement: 50%, 33178.33 CHF
Déduction forfaitaire si CTAA <=50%: 10.0%, 29860.5 CHF

Préjudice économique:
90933.96226415

# Chatgpt function calling

In [202]:
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

True

In [203]:
query = """<context>
    L'utilisateur intéragit avec le copilote en lui soumettant des cas de décision de rente assurance. Les cas doivent être reformulé en donné précise afin d'être utilisé pour calculer la rente.
</context>

<objectif>
    Ecrire des informations claires et précises afin de les réutiliser dans une fonction permettant de déterminer la somme de la rente pour un demandeur de rente assurance invalidité.
</objectif>

<instructions>
    <instruction>Étant donné la déscription donner par l'utilisateur, extractez les informations pertinentes au <calcul></instruction>
    <instruction>L'extraction d'informations suis un processus spécifique: déterminer le sexe du demandeur, la branche économique, le niveau de compétence, le salaire ofs et l'année correspondante, la date d'exigibilité, le salaire effectif et l'année qui correspond (optionnel) , le taux d'activité, la diminution du taux après l'atteinte à la santé et le salaire avant l'atteinte à la santée ainsi que l'année correspondante (optionnel)</instruction>
    <instruction>Le niveau de compétence est décris comme suis: 4 = Tâches qui exigent une capacité à résoudre des problèmes complexes et à prendre des décisions fondées sur un vaste ensemble de connaissances théoriques et factuelles dans un domaine spécialisé. 3 = Tâches pratiques complexes nécessitant un vaste ensemble de connaissances dans un domaine spécialisé. 2 = Tâches pratiques telles que la vente/ les soins/ le traitement de données et les tâches administratives/ l'utilisation de machines et d'appareils électroniques/ les services de sécurité/ la conduite de véhicules. 1 = Tâches physiques ou manuelles simples.</instruction>
    <instruction> Si une information optionnel n'est pas présente dans la déscription du cas la valeur a attribuer est de 0</instruction>
    <instruction>Formulez les informations extraites sous forme d'objet python</instruction>

<format_de_réponse>
beneficiary = {
    "sexe": str,            # le sexe du demandeur. valeurs acceptée: ["homme","femme","26 al. 6 RAI"]
    "branche": str,         # la branche économique.
    "niveau_comp": int,     # le niveau de compétence
    "salaire_ofs": {
        "année": int,       # année de référence
        "salaire": double   # salaire OFS
    },
    "ess": int,             # année d'exigibilité
    "salaire_effectif": {
        "salaire": int,     # salaire effectif
        "année": double     # année correspondante
    },
    "horaire": int,         # taux d'activité. valeur entre 0-100
    "diminution": int,      # réduction du taux d'activité. valeur entre 0-100
    "salaire_as": {
        "salaire": double   # salaire avant l'atteinte à la santé
        "année": int        # année correspondante
    }
}
</format_de_réponse>

<exemples>
Activité avant atteinte à la santé : Ouvrier de la construction qualifié (CFC)
Taux d’activité : 100% / Revenu sans invalidité effectif 2021: CHF 90'000.00
Activité exigible avec invalidité : Activité légère non qualifiée dans la production ou les services (05-96)
salaire ofs: 5305.00
Taux d’activité : 50%
Date de l’exigibilité : 01.07.2022 -> beneficiary = {
    "sexe": "homme",
    "branche": "41-43",
    "niveau_comp": 1,
    "salaire_ofs": {
        "année": 2022,
        "salaire": 5305
    },
    "ess": 2022,
    "salaire_effectif": {
        "salaire": 0,
        "année": 0
    },
    "horaire": 100,
    "diminution": 50,
    "salaire_as": {
        "salaire": 90000,
        "année": 2021
    }
}
</exemples>
"""

In [208]:
client = OpenAI()

tools = [{
    "type": "function",
    "name": "mise_en_parallele_des_revenus",
    "description": "Compute the revenues of the AI beneficiary.",
    "parameters": {
        "type": "object",
        "properties": {
            "beneficiary": {
                "type": "object"
            }
        },
        "required": ["beneficiary"],
        "additionalProperties": False
    },
    "strict": True
}]

input_messages = [
    # {"role": "system", "content": query},
    {"role": "user", "content": "What's the weather like in Paris today?"}
]

response = client.chat.completions.create(
    model="gpt-4o",
    messages=input_messages,
)

APITimeoutError: Request timed out.

In [ ]:
tool_call = completion.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)

result = get_weather(args["latitude"], args["longitude"])